# RetinaVessel AI — multi-seed replication v5 (seed 123)

This notebook independently fits the standard U-Net, clean lightweight U-Net and degradation-trained lightweight U-Net with training RNG seed 123. The audited image split remains frozen. After training, it executes the complete validation-only v3 robustness/topology matrix.

**Protocol boundary:** seed 123 was predeclared with seeds 7 and 42. Do not alter model settings based on seed-42 or seed-7 results, and do not inspect or evaluate the official FIVES test images. Enable a GPU accelerator.

In [ ]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/simlexx-k/retinavessel-ai.git"
EXPECTED_COMMIT = "224e115808a3330ee17516f0348ccd8fda9aa5ba"
REPO = Path("/kaggle/working/retinavessel-ai-v5-seed123-code")
if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements.txt")], check=True)
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip()
assert actual_commit == EXPECTED_COMMIT
os.chdir(REPO)
print("Pinned replication code:", actual_commit)

In [ ]:
import torch

sys.path.insert(0, str(REPO / "src"))
from retinavessel.data import find_fives_root, load_manifest, samples_from_frozen_split

assert torch.cuda.is_available(), "Enable a GPU accelerator before running v5"
DATA_MOUNT = Path("/kaggle/input/datasets/simonkosgei/fives-retinal-dataset")
assert DATA_MOUNT.exists(), f"Missing attached FIVES dataset: {DATA_MOUNT}"
FIVES_ROOT = find_fives_root(DATA_MOUNT)
SPLIT_MANIFEST = REPO / "splits/fives_seed42.json"
EXPECTED_SPLIT_SHA256 = "aee8809176974d6f880013d85a3ab4a405a43121d0cf31363fcc47dd75256c87"
assert hashlib.sha256(SPLIT_MANIFEST.read_bytes()).hexdigest() == EXPECTED_SPLIT_SHA256
split = json.loads(SPLIT_MANIFEST.read_text())
train_samples, validation_samples = samples_from_frozen_split(load_manifest(FIVES_ROOT, "train"), split)
assert len(train_samples) == 504 and len(validation_samples) == 89
assert len(load_manifest(FIVES_ROOT, "test")) == 200  # count only; no pixels opened
print("PyTorch:", torch.__version__)
print("CUDA device:", torch.cuda.get_device_name(0))
print("Frozen split: 504 train / 89 validation / 7 excluded")
print("Official test images inspected or evaluated: 0")
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

In [ ]:
test_environment = os.environ.copy()
test_environment["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), str(REPO / "backend")])
subprocess.run([sys.executable, "-m", "pytest", "-q"], cwd=REPO, env=test_environment, check=True)

In [ ]:
MAX_EPOCHS = 30
PATIENCE = 5
SEED = 123
RUN_TIMES = {}
EXPERIMENTS = [
    {"name": "unet_seed123_es", "model": "unet", "degradation": False},
    {"name": "lite_unet_seed123_es", "model": "lite_unet", "degradation": False},
    {"name": "lite_robust_seed123_es", "model": "lite_unet", "degradation": True},
]

def run_experiment(spec):
    output = REPO / "outputs" / spec["name"]
    last_checkpoint = output / "last.pt"
    command = [
        sys.executable, "-m", "retinavessel.train",
        "--data", str(DATA_MOUNT), "--model", spec["model"],
        "--epochs", str(MAX_EPOCHS), "--patience", str(PATIENCE),
        "--batch-size", "2", "--patch-size", "384",
        "--patches-per-image", "2", "--overlap", "64",
        "--split-manifest", str(SPLIT_MANIFEST),
        "--seed", str(SEED), "--device", "cuda",
        "--out", str(output),
    ]
    if spec["degradation"]:
        command.append("--degradation")
    if last_checkpoint.exists():
        state = torch.load(last_checkpoint, map_location="cpu", weights_only=False)
        complete = int(state["epoch"]) >= MAX_EPOCHS or int(state["epochs_without_improvement"]) >= PATIENCE
        if complete:
            print(f"{spec['name']} already complete at epoch {state['epoch']}; skipping")
            return
        command.extend(["--resume", str(last_checkpoint)])
    environment = os.environ.copy()
    environment["PYTHONPATH"] = str(REPO / "src")
    started = time.perf_counter()
    subprocess.run(command, cwd=REPO, env=environment, check=True)
    RUN_TIMES[spec["name"]] = time.perf_counter() - started
    for required in ("best.pt", "last.pt", "history.json", "split_manifest.json"):
        assert (output / required).is_file(), f"Missing {output / required}"

def save_partial_snapshot(label):
    target = Path(f"/kaggle/working/retinavessel_v5_seed123_{label}")
    archive = Path(shutil.make_archive(str(target), "zip", root_dir=REPO, base_dir="outputs"))
    print("Partial recovery archive:", archive)

## E1 replication — standard U-Net, seed 123

In [ ]:
run_experiment(EXPERIMENTS[0])
save_partial_snapshot("after_unet")

## E2 replication — clean lightweight U-Net, seed 123

In [ ]:
run_experiment(EXPERIMENTS[1])
save_partial_snapshot("after_lite")

## E3 replication — degradation-trained lightweight U-Net, seed 123

In [ ]:
run_experiment(EXPERIMENTS[2])
save_partial_snapshot("training_complete")

In [ ]:
import pandas as pd
from retinavessel.model import UNet, model_params

rows = []
for spec in EXPERIMENTS:
    output = REPO / "outputs" / spec["name"]
    history = json.loads((output / "history.json").read_text())
    checkpoint_path = output / "best.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    assert checkpoint["seed"] == SEED
    network = UNet(base=int(checkpoint["base"]), lite=checkpoint["architecture"] == "lite_unet")
    network.load_state_dict(checkpoint["model_state"], strict=True)
    rows.append({
        "experiment": spec["name"], "model": checkpoint["architecture"],
        "degradation_training": spec["degradation"], "seed": SEED,
        "parameters": model_params(network), "completed_epochs": history[-1]["epoch"],
        "best_epoch": checkpoint["epoch"], "best_validation_dice": checkpoint["validation_dice"],
        "checkpoint_sha256": hashlib.sha256(checkpoint_path.read_bytes()).hexdigest(),
        "wall_minutes_this_notebook": round(RUN_TIMES.get(spec["name"], float("nan")) / 60, 2),
    })
summary = pd.DataFrame(rows).sort_values("best_validation_dice", ascending=False)
display(summary)
SUMMARY_DIR = REPO / "outputs" / "v5_seed123_summary"
SUMMARY_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(SUMMARY_DIR / "validation_summary.csv", index=False)
run_manifest = {
    "training_code_commit": EXPECTED_COMMIT, "split_manifest_sha256": EXPECTED_SPLIT_SHA256,
    "pytorch_version": torch.__version__, "cuda_device": torch.cuda.get_device_name(0),
    "maximum_epochs": MAX_EPOCHS, "early_stopping_patience": PATIENCE,
    "seed": SEED, "official_test_evaluated": False,
    "results": json.loads(summary.to_json(orient="records")),
}
(SUMMARY_DIR / "run_manifest.json").write_text(json.dumps(run_manifest, indent=2))

## Full seed-123 validation matrix

This is the same 3 × 9 × 89 validation-only matrix used for seed 42. It performs no model selection and never accesses official-test pixels.

In [ ]:
VALIDATION_OUTPUT = REPO / "outputs" / "validation_v5_seed123"
evaluation_command = [
    sys.executable, "-m", "retinavessel.validation",
    "--data", str(DATA_MOUNT), "--split-manifest", str(SPLIT_MANIFEST),
    "--expected-split-sha256", EXPECTED_SPLIT_SHA256,
    "--model", f"unet_seed123={REPO / 'outputs/unet_seed123_es/best.pt'}",
    "--model", f"lite_unet_seed123={REPO / 'outputs/lite_unet_seed123_es/best.pt'}",
    "--model", f"lite_robust_seed123={REPO / 'outputs/lite_robust_seed123_es/best.pt'}",
    "--topology", "--device", "cuda", "--warmup-runs", "2",
    "--degradation-seed", "2026", "--bootstrap-seed", "2026",
    "--bootstrap-repetitions", "2000", "--out", str(VALIDATION_OUTPUT),
]
evaluation_environment = os.environ.copy()
evaluation_environment["PYTHONPATH"] = str(REPO / "src")
subprocess.run(evaluation_command, cwd=REPO, env=evaluation_environment, check=True)
report = json.loads((VALIDATION_OUTPUT / "validation_report.json").read_text())
per_image = pd.read_csv(VALIDATION_OUTPUT / "validation_per_image.csv")
assert report["official_test_evaluated"] is False
assert report["validation_image_count"] == 89 and report["debug_validation_limit"] is None
assert len(per_image) == 3 * 9 * 89
display(pd.read_csv(VALIDATION_OUTPUT / "validation_summary.csv"))

In [ ]:
ARCHIVE_BASE = Path("/kaggle/working/retinavessel_v5_seed123")
archive = Path(shutil.make_archive(str(ARCHIVE_BASE), "zip", root_dir=REPO, base_dir="outputs"))
print(f"Final artifact archive: {archive} ({archive.stat().st_size / 1024**2:.1f} MiB)")
print("Save this notebook version and download the archive before aggregation.")

## Stop here

Do not open the official FIVES test partition. Download and verify the seed-123 archive, then aggregate all three predeclared seeds. Only after aggregation may the model configuration and final test policy be frozen.